# Distance to Nearest Pharmacy from SAL Centroids
## Euclidean and OSMnx Network Distance (Pedestrian + Drive)

This notebook computes the distance from each Small Area Layer (SAL) centroid
to the nearest geocoded pharmacy using three methods:

1. Euclidean (straight-line) distance via scipy KDTree.
2. Pedestrian network distance via `OSMnx` walk graph.
3. Drive network distance via `OSMnx` drive graph.

Network graphs are saved as `.graphml` files for reuse.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from scipy.spatial import cKDTree
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib as mpl
import mapclassify
import warnings
import os
import time

warnings.filterwarnings("ignore")
print(f"OSMnx Version: {ox.__version__}")

OSMnx Version: 2.1.0


## CONFIGURATION
Update file paths below to match local directory structure.
All paths are relative to the notebook location, following the
partner notebook conventions.

In [5]:
# FILE PATHS
# SAL shapefile with ward assignments and geometry.
SAL_SHP_PATH = "data/sal_w_ward_new/sal_w_ward_new.shp"

# Population estimates from the step-down and dasymetric notebook.
POP_ESTIMATES_PATH = "data/pop_pred_final.csv"

# Fully geocoded pharmacy file with lat and lng columns.
PHARMACIES_PATH = "data/PHARMACIES_places_full_results.csv"

# OUTPUT PATHS
# Directory for saved network graphs and results.
OUTPUT_DIR = "data/networks"
os.makedirs(OUTPUT_DIR, exist_ok = True)

# Projected CRS for distance calculations in meters.
# South African standard: Hartebeesthoek94 / Lo29 for central SA.
PROJECTED_CRS = "EPSG:32735"

# Province names for filtering and labeling.
PROVINCES = ["Gauteng", "KwaZulu-Natal"]

print(f"Output Directory: {os.path.abspath(OUTPUT_DIR)}")

Output Directory: c:\Users\Tess\Desktop\UPenn\UPenn_SS26\MUSA_8010-001_Practicum\south-africa-healthcare\notebooks\data\networks


## LOAD SAL GEOMETRIES AND POPULATION ESTIMATES
Load the SAL shapefile that has ward assignments, then join
with the population estimates produced by the dasymetric notebook.

In [6]:
# Load SAL shapefile with geometry and ward assignments.
sal_geo = gpd.read_file(SAL_SHP_PATH)
print(f"SAL shapefile loaded: {sal_geo.shape[0]} features")
print(f"CRS: {sal_geo.crs}")
print(f"Columns: {list(sal_geo.columns)}")

SAL shapefile loaded: 39177 features
CRS: EPSG:32735
Columns: ['SP_CODE', 'SP_NAME', 'MP_CODE', 'MP_NAME', 'MN_MDB_C', 'MN_CODE', 'MN_NAME', 'MN_TYPE', 'DC_MDB_C', 'DC_MN_C', 'DC_NAME', 'PR_MDB_C', 'PR_CODE', 'PR_NAME', 'EA_GTYPE', 'ALBERS_ARE', 'MD_CODE', 'MD_NAME', 'Shape_Leng', 'SAL_CODE', 'EA_TYPE', 'F4_class', 'EA_area_km', 'num_houses', 'F4_class_2', 'num_build', 'EA_CODE_1', 'old_EA_TYP', 'smallplace', 'url', 'Black_Afri', 'White', 'Coloured', 'Indian_or', 'Other', 'population', 'F0_4', 'F5_9', 'F10_14', 'F15_19', 'F20_24', 'F25_29', 'F30_34', 'F35_39', 'F40_44', 'F45_49', 'F50_54', 'F55_59', 'F60_64', 'F65_69', 'F70_74', 'F75_79', 'F80_84', 'F85_', 'Shape_Le_1', 'Shape_Area', 'sal_pop_de', 'sal2011_po', 'OBJECTID_1', 'EA_CODE', 'census_war', 'AREA', 'PERCENTAGE', 'OBJECTID_2', 'EA_CODE_12', 'OBJECTID_3', 'FREQUENCY', 'EA_CODE_13', 'MAX_AREA', 'geometry']


In [7]:
# Load population estimates from the dasymetric step-down.
pop_est = pd.read_csv(POP_ESTIMATES_PATH)
print(f"Population estimates loaded: {pop_est.shape[0]} rows")
print(f"Columns: {list(pop_est.columns)}")

Population estimates loaded: 38380 rows
Columns: ['WardID', 'EA_CODE', 'sal2011_pop', 'ward2023_pop', 'EA_GTYPE', 'EA_TYPE', 'econ_status', 'houses2011', 'Black_Afri', 'White', 'Coloured', 'Indian_or', 'Other', 'area_km2', 'sal_dense', 'log_density', 'ward2011_sum', 'share2011', 'dasym_weight', 'sal2023_est', 'growth_rate']


In [8]:
# Join population estimates onto SAL geometries.
# The join key is EA_CODE (SAL identifier).
sal = sal_geo.merge(
    pop_est[["EA_CODE", "sal2023_est", "WardID"]],
    on = "EA_CODE",
    how = "left"
)

# Report join quality.
matched = sal["sal2023_est"].notna().sum()
total = sal.shape[0]
print(f"JOIN RESULTS: {matched}/{total} SALs matched ({matched/total*100:.1f}%)")

# Drop SALs with no population estimate.
sal = sal[sal["sal2023_est"].notna()].copy()
print(f"SALs retained after dropping unmatched: {sal.shape[0]}")

JOIN RESULTS: 39177/39177 SALs matched (100.0%)
SALs retained after dropping unmatched: 39177


## COMPUTE SAL CENTROIDS
Project to a metric CRS, compute geometric centroids, then extract
coordinates for distance calculations.

**Limitation:** Geometric centroids may fall in uninhabited areas
(rivers, parks, industrial zones) for irregularly shaped SALs.
Population-weighted centroids using building footprints would be
more defensible but require additional data processing.

In [9]:
# Project SAL geometries to metric CRS for distance calculations.
sal_proj = sal.to_crs(PROJECTED_CRS)

# Compute geometric centroids in projected space.
sal_proj["centroid_geom"] = sal_proj.geometry.centroid

# Extract centroid coordinates in projected CRS (meters).
sal_proj["centroid_x"] = sal_proj["centroid_geom"].x
sal_proj["centroid_y"] = sal_proj["centroid_geom"].y

# Also compute centroids in WGS84 for OSMnx (which expects lat/lng).
sal_wgs84 = sal.to_crs("EPSG:4326")
sal_wgs84["centroid_wgs84"] = sal_wgs84.geometry.centroid
sal_proj["centroid_lat"] = sal_wgs84["centroid_wgs84"].y.values
sal_proj["centroid_lng"] = sal_wgs84["centroid_wgs84"].x.values

print(f"SAL CENTROIDS COMPUTED: {sal_proj.shape[0]} centroids")
print(f"Projected CRS: {PROJECTED_CRS}")
print(f"Centroid X range: {sal_proj['centroid_x'].min():.0f} to {sal_proj['centroid_x'].max():.0f} m")
print(f"Centroid Y range: {sal_proj['centroid_y'].min():.0f} to {sal_proj['centroid_y'].max():.0f} m")

SAL CENTROIDS COMPUTED: 39177 centroids
Projected CRS: EPSG:32735
Centroid X range: 525554 to 1084240 m
Centroid Y range: 6557451 to 7210624 m


## LOAD GEOCODED PHARMACIES
Load the fully geocoded pharmacy file and convert to a GeoDataFrame.

In [10]:
# Load geocoded pharmacies.
pharm_df = pd.read_csv(PHARMACIES_PATH)
print(f"Pharmacies loaded: {pharm_df.shape[0]} rows")
print(f"Columns: {list(pharm_df.columns)}")

# Check for lat/lng columns and filter valid coordinates.
assert "lat" in pharm_df.columns, "Missing 'lat' column in pharmacy file."
assert "lng" in pharm_df.columns, "Missing 'lng' column in pharmacy file."

# Drop rows with missing coordinates.
valid_coords = pharm_df["lat"].notna() & pharm_df["lng"].notna()
print(f"Pharmacies with valid coordinates: {valid_coords.sum()}/{pharm_df.shape[0]}")
pharm_df = pharm_df[valid_coords].copy()

# Convert to GeoDataFrame.
pharm_gdf = gpd.GeoDataFrame(
    pharm_df,
    geometry = gpd.points_from_xy(pharm_df["lng"], pharm_df["lat"]),
    crs = "EPSG:4326"
)

# Project to metric CRS.
pharm_proj = pharm_gdf.to_crs(PROJECTED_CRS)
pharm_proj["pharm_x"] = pharm_proj.geometry.x
pharm_proj["pharm_y"] = pharm_proj.geometry.y

print(f"PHARMACIES READY: {pharm_proj.shape[0]} geocoded locations")

Pharmacies loaded: 4812 rows
Columns: ['raw_row_id', 'PHARMACY_ID', 'PRACTICE_NUM', 'PRACTICE_NAME', 'ADDRESS', 'CITY', 'PROVINCE', 'PHONE', 'query_string', 'http_status', 'api_status', 'place_id', 'matched_name', 'matched_address', 'lat', 'lng', 'types', 'raw_error', 'needs_review']
Pharmacies with valid coordinates: 4539/4812
PHARMACIES READY: 4539 geocoded locations


In [11]:
# Quick sanity check: bounding box of pharmacies vs SALs.
print("BOUNDING BOX COMPARISON (WGS84)")
sal_bounds = sal.to_crs("EPSG:4326").total_bounds
pharm_bounds = pharm_gdf.total_bounds
print(f"SALs:       W={sal_bounds[0]:.3f}, S={sal_bounds[1]:.3f}, E={sal_bounds[2]:.3f}, N={sal_bounds[3]:.3f}")
print(f"Pharmacies: W={pharm_bounds[0]:.3f}, S={pharm_bounds[1]:.3f}, E={pharm_bounds[2]:.3f}, N={pharm_bounds[3]:.3f}")

BOUNDING BOX COMPARISON (WGS84)
SALs:       W=27.156, S=-31.083, E=32.891, N=-25.110
Pharmacies: W=-123.090, S=-34.091, E=77.284, N=50.943


## EUCLIDEAN DISTANCE TO NEAREST PHARMACY
Use a KDTree for efficient nearest-neighbor lookup in projected
coordinates (meters). This gives straight-line distances that
underestimate true travel distances but serve as a useful lower bound.

In [12]:
# Build KDTree from pharmacy locations in projected CRS.
pharm_coords = np.column_stack([pharm_proj["pharm_x"].values, pharm_proj["pharm_y"].values])
tree = cKDTree(pharm_coords)

# Query nearest pharmacy for each SAL centroid.
sal_coords = np.column_stack([sal_proj["centroid_x"].values, sal_proj["centroid_y"].values])
distances_m, indices = tree.query(sal_coords, k = 1)

# Store results.
sal_proj["euclidean_dist_m"] = distances_m
sal_proj["euclidean_dist_km"] = distances_m / 1000.0
sal_proj["nearest_pharm_idx"] = indices

print("EUCLIDEAN DISTANCE COMPUTED")
print(sal_proj["euclidean_dist_km"].describe().to_string())

EUCLIDEAN DISTANCE COMPUTED
count    39177.000000
mean         4.721430
std          6.983172
min          0.005177
25%          0.788219
50%          1.641133
75%          4.809992
max         50.162386


## EUCLIDEAN DISTANCE HEATMAP
Choropleth map of SAL polygons colored by Euclidean distance
to the nearest pharmacy, shown per province.

In [13]:
# Prepare data for mapping (keep original geometry, not centroids).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

# Determine province column name from shapefile.
# Common variants: PR_NAME, PROVINCE, Province.
prov_col = None
for candidate in ["PR_NAME", "PROVINCE", "Province", "province"]:
    if candidate in sal_map_data.columns:
        prov_col = candidate
        break

if prov_col is None:
    print("WARNING: No province column found. Plotting all SALs together.")
    prov_col = "ALL"
    sal_map_data["ALL"] = "All Provinces"

print(f"Province column: {prov_col}")
print(f"Unique values: {sal_map_data[prov_col].unique()}")

Province column: PR_NAME
Unique values: <ArrowStringArray>
['KwaZulu-Natal', 'Gauteng']
Length: 2, dtype: str


In [ ]:
# Define distance bins in kilometers.
dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]

fig, axes = plt.subplots(1, 2, figsize = (18, 10))
cmap = mpl.cm.RdYlGn_r

provinces_in_data = sal_map_data[prov_col].unique()

for i, prov in enumerate(provinces_in_data[:2]):
    subset = sal_map_data[sal_map_data[prov_col] == prov].copy()
    subset = subset[subset["euclidean_dist_km"].notna()]

    subset.plot(
        column = "euclidean_dist_km",
        cmap = cmap,
        scheme = "UserDefined",
        classification_kwds = {"bins": dist_bins_km},
        legend = True,
        legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fontsize": 8},
        ax = axes[i],
        edgecolor = "none",
        linewidth = 0
    )

    axes[i].set_title(f"{prov}: Euclidean Distance to Nearest Pharmacy", fontsize = 12)
    axes[i].axis("off")

plt.suptitle("SAL Centroid Euclidean Distance to Nearest Pharmacy", fontsize = 14, y = 1.02)
plt.tight_layout()

output_path = os.path.join(OUTPUT_DIR, "images/sal_pharmacy_distance/heatmap_euclidean_distance.png")
plt.savefig(output_path, dpi = 300, bbox_inches = "tight")
plt.show()
print(f"Map saved to {output_path}")

## DOWNLOAD OSMnx NETWORK GRAPHS
Download pedestrian (walk) and drive networks for each province.
These are large downloads; each graph is saved as `.graphml`
for reuse without re-downloading.

**Note:** This step may take 10-30+ minutes per province depending
on network size and connection speed. The saved `.graphml` files
can be reloaded in future sessions.

In [15]:
# Configure OSMnx settings for large downloads.
ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.timeout = 600

# Define place queries for each province.
place_queries = {
    "Gauteng": "Gauteng, South Africa",
    "KwaZulu-Natal": "KwaZulu-Natal, South Africa"
}

# Network types to download.
network_types = {
    "walk": "pedestrian",
    "drive": "drive"
}

# Use cache because files are large.
print(f"Cache Enabled: {ox.settings.use_cache}")
print(f"Timeout: {ox.settings.timeout}s")

Cache Enabled: True
Timeout: 600s


In [16]:
# Download and save network graphs.
# Store graph references for distance computation.
graphs = {}

for prov_name, place_query in place_queries.items():
    for net_type, net_label in network_types.items():
        graph_key = f"{prov_name}_{net_type}"
        graphml_path = os.path.join(OUTPUT_DIR, f"network_{prov_name.lower().replace('-', '_')}_{net_type}.graphml")

        # Check if graph already exists on disk.
        if os.path.exists(graphml_path):
            print(f"CACHED GRAPH: {graph_key}")
            G = ox.load_graphml(graphml_path)
        else:
            print(f"DOWNLOADING {net_label.upper()} NETWORK: {place_query}")
            start_time = time.time()
            G = ox.graph_from_place(place_query, network_type = net_type)
            elapsed = time.time() - start_time
            print(f"Download complete in {elapsed:.1f} seconds.")

            # Add edge lengths if not already present.
            if not all("length" in data for _, _, data in G.edges(data = True)):
                G = ox.distance.add_edge_lengths(G)

            # Save to disk.
            ox.save_graphml(G, graphml_path)
            print(f"Graph saved to {graphml_path}")

        # Store reference.
        graphs[graph_key] = G
        print(f"  Nodes: {G.number_of_nodes():,}, Edges: {G.number_of_edges():,}\n")

print(f"Graph Keys: {list(graphs.keys())}")

DOWNLOADING PEDESTRIAN NETWORK: Gauteng, South Africa
Download complete in 1919.5 seconds.
Graph saved to data/networks\network_gauteng_walk.graphml
  Nodes: 443,832, Edges: 1,219,312

DOWNLOADING DRIVE NETWORK: Gauteng, South Africa
Download complete in 1579.0 seconds.
Graph saved to data/networks\network_gauteng_drive.graphml
  Nodes: 279,004, Edges: 730,792

DOWNLOADING PEDESTRIAN NETWORK: KwaZulu-Natal, South Africa
Download complete in 5312.4 seconds.
Graph saved to data/networks\network_kwazulu_natal_walk.graphml
  Nodes: 541,204, Edges: 1,356,318

DOWNLOADING DRIVE NETWORK: KwaZulu-Natal, South Africa
Download complete in 4826.8 seconds.
Graph saved to data/networks\network_kwazulu_natal_drive.graphml
  Nodes: 311,426, Edges: 749,194

Graph Keys: ['Gauteng_walk', 'Gauteng_drive', 'KwaZulu-Natal_walk', 'KwaZulu-Natal_drive']


## NETWORK DISTANCE COMPUTATION
For each network graph, compute the shortest-path distance from
every SAL centroid to the nearest pharmacy.

**Strategy:** Use multi-source Dijkstra from all pharmacy nodes
simultaneously. This computes the shortest distance from *any*
pharmacy to every reachable node in the graph in a single pass,
which is far more efficient than routing from each SAL centroid
individually.

**Steps:**
1. Snap all pharmacy locations to nearest network nodes.
2. Snap all SAL centroids to nearest network nodes.
3. Run multi-source Dijkstra from all pharmacy nodes.
4. Look up the distance for each SAL centroid node.

In [ ]:
def compute_network_distances(G, sal_df, pharm_df, label = "network"):
    """
    Compute shortest-path distance from each SAL centroid to the
    nearest pharmacy using multi-source Dijkstra on the given graph.

    Parameters
    G : networkx.MultiDiGraph
        OSMnx network graph (unprojected, WGS84).
    sal_df : GeoDataFrame
        SAL data with centroid_lat and centroid_lng columns.
    pharm_df : GeoDataFrame
        Pharmacy data with lat and lng columns.
    label : str
        Label for the distance column.

    Returns
    np.ndarray
        Array of distances in meters, one per SAL centroid.
        NaN for unreachable centroids.
    """

    # Snap pharmacies to nearest network nodes.
    pharm_nodes = ox.nearest_nodes(
        G,
        X = pharm_df["lng"].values,
        Y = pharm_df["lat"].values
    )
    unique_pharm_nodes = list(set(pharm_nodes))
    print(f"  Pharmacy nodes snapped: {len(pharm_nodes)} -> {len(unique_pharm_nodes)} unique.")

    # Snap SAL centroids to nearest network nodes.
    sal_nodes = ox.nearest_nodes(
        G,
        X = sal_df["centroid_lng"].values,
        Y = sal_df["centroid_lat"].values
    )
    print(f"  SAL Centroid Nodes Snapped: {len(sal_nodes)}.")

    # Multi-source Dijkstra from all pharmacy nodes.
    # Convert to undirected for walk networks to allow bidirectional traversal.
    G_undirected = G.to_undirected()

    print(f"  Multi-source Dijkstra ({label}).")
    start_time = time.time()
    dist_dict = nx.multi_source_dijkstra_path_length(
        G_undirected,
        sources = set(unique_pharm_nodes),
        weight = "length"
    )
    elapsed = time.time() - start_time
    print(f"  Dijkstra complete in {elapsed:.1f} seconds, {len(dist_dict):,} nodes reached.")

    # Step 4: Look up distance for each SAL centroid node.
    distances = np.array([
        dist_dict.get(node, np.nan) for node in sal_nodes
    ])

    reachable = np.isfinite(distances).sum()
    print(f"  Reachable SAL Centroids: {reachable}/{len(distances)} ({reachable/len(distances)*100:.1f}%)")

    return distances

In [ ]:
# Compute network distances for each province and network type.
# Filter SALs and pharmacies by province before computing.
for prov_name in place_queries.keys():
    # Filter SALs for this province.
    prov_mask_sal = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
    sal_prov = sal_proj[prov_mask_sal].copy()

    if sal_prov.shape[0] == 0:
        print(f"WARNING: No SALs for {prov_name}. Check province column values.")
        continue

    # Filter pharmacies for this province.
    # Try common column names for province in pharmacy data.
    pharm_prov_col = None
    for candidate in ["PROVINCE", "Province", "province"]:
        if candidate in pharm_gdf.columns:
            pharm_prov_col = candidate
            break

    if pharm_prov_col is not None:
        prov_mask_pharm = pharm_gdf[pharm_prov_col].str.contains(prov_name, case = False, na = False)
        pharm_prov = pharm_gdf[prov_mask_pharm].copy()
    else:
        # If no province column, use spatial join to filter pharmacies within province bounds.
        prov_bounds = sal_prov.to_crs("EPSG:4326").total_bounds
        pharm_prov = pharm_gdf.cx[prov_bounds[0]:prov_bounds[2], prov_bounds[1]:prov_bounds[3]].copy()

    print(f"PROVINCE: {prov_name}")
    print(f"  SALs: {sal_prov.shape[0]}, Pharmacies: {pharm_prov.shape[0]}")

    for net_type in network_types.keys():
        graph_key = f"{prov_name}_{net_type}"
        if graph_key not in graphs:
            print(f"  SKIPPING {graph_key}: Graph not loaded.")
            continue

        G = graphs[graph_key]
        col_name = f"{net_type}_dist_m"
        col_name_km = f"{net_type}_dist_km"

        print(f"  Computing {net_type} network distances.")
        distances = compute_network_distances(
            G = G,
            sal_df = sal_prov,
            pharm_df = pharm_prov,
            label = net_type
        )

        # Store results back in the main dataframe.
        sal_proj.loc[prov_mask_sal, col_name] = distances
        sal_proj.loc[prov_mask_sal, col_name_km] = distances / 1000.0

PROVINCE: Gauteng
  SALs: 21182, Pharmacies: 2879
  Computing walk network distances.
  Pharmacy nodes snapped: 2879 -> 1090 unique.
  SAL Centroid Nodes Snapped: 21182.
  Multi-source Dijkstra (walk).
  Dijkstra complete in 22.6 seconds, 443,832 nodes reached.
  Reachable SAL Centroids: 21182/21182 (100.0%)
  Computing drive network distances.
  Pharmacy nodes snapped: 2879 -> 1061 unique.
  SAL Centroid Nodes Snapped: 21182.
  Multi-source Dijkstra (drive).
  Dijkstra complete in 3.2 seconds, 279,004 nodes reached.
  Reachable SAL Centroids: 21182/21182 (100.0%)
PROVINCE: KwaZulu-Natal
  SALs: 17995, Pharmacies: 1660
  Computing walk network distances.
  Pharmacy nodes snapped: 1660 -> 610 unique.
  SAL Centroid Nodes Snapped: 17995.
  Multi-source Dijkstra (walk).
  Dijkstra complete in 5.3 seconds, 541,204 nodes reached.
  Reachable SAL Centroids: 17995/17995 (100.0%)
  Computing drive network distances.
  Pharmacy nodes snapped: 1660 -> 575 unique.
  SAL Centroid Nodes Snapped: 17

## DISTANCE SUMMARY STATISTICS
Compare Euclidean, pedestrian, and drive distances across provinces.

In [19]:
# Summary statistics by province.
distance_cols = ["euclidean_dist_km", "walk_dist_km", "drive_dist_km"]
available_cols = [c for c in distance_cols if c in sal_proj.columns]

for prov_name in place_queries.keys():
    prov_mask = sal_proj[prov_col].str.contains(prov_name, case = False, na = False)
    subset = sal_proj[prov_mask]

    print(f"DISTANCE STATISTICS: {prov_name.upper()}")
    print(f"SALs: {subset.shape[0]}")
    for col in available_cols:
        print(f"  {col}:")
        print(subset[col].describe().to_string())
        print()

# Circuity ratios (network / Euclidean).
for net_col in ["walk_dist_km", "drive_dist_km"]:
    if net_col in sal_proj.columns:
        ratio_col = net_col.replace("_dist_km", "_circuity")
        sal_proj[ratio_col] = sal_proj[net_col] / sal_proj["euclidean_dist_km"]
        median_ratio = sal_proj[ratio_col].median()
        print(f"MEDIAN CIRCUITY RATIO ({net_col}): {median_ratio:.2f}")

DISTANCE STATISTICS: GAUTENG
SALs: 21182
  euclidean_dist_km:
count    21182.000000
mean         1.820853
std          2.203106
min          0.008063
25%          0.637851
50%          1.160122
75%          2.101947
max         35.518173

  walk_dist_km:
count    21182.000000
mean         2.484196
std          2.577650
min          0.000000
25%          0.965070
50%          1.685913
75%          2.964315
max         41.518936

  drive_dist_km:
count    21182.000000
mean         2.462142
std          2.601755
min          0.000000
25%          0.919676
50%          1.639156
75%          2.990041
max         38.500049

DISTANCE STATISTICS: KWAZULU-NATAL
SALs: 17995
  euclidean_dist_km:
count    17995.000000
mean         8.135712
std          8.882217
min          0.005177
25%          1.237684
50%          3.828475
75%         13.776844
max         50.162386

  walk_dist_km:
count    17995.000000
mean        11.562804
std         12.736088
min          0.000000
25%          1.885383
50%

## EXPORT RESULTS
Save the SAL-level distance results for downstream analysis
and app integration.

In [22]:
# Select columns to export.
export_cols = ["EA_CODE", "WardID", "sal2023_est",
               "centroid_lat", "centroid_lng",
               "euclidean_dist_m", "euclidean_dist_km"]

# Add network distance columns if available.
for col in ["walk_dist_m", "walk_dist_km", "drive_dist_m", "drive_dist_km",
            "walk_circuity", "drive_circuity"]:
    if col in sal_proj.columns:
        export_cols.append(col)

# Add province column.
if prov_col in sal_proj.columns:
    export_cols.insert(1, prov_col)

# Filter to only columns that exist.
export_cols = [c for c in export_cols if c in sal_proj.columns]

export_df = sal_proj[export_cols].copy()
export_path = os.path.join(OUTPUT_DIR, "sal_pharmacy_distances.csv")
export_df.to_csv(export_path, index = False)

print(f"RESULTS EXPORTED: {export_path}")
print(f"Rows: {export_df.shape[0]}, Columns: {export_df.shape[1]}")
print(f"Columns: {list(export_df.columns)}")

RESULTS EXPORTED: data/networks\sal_pharmacy_distances.csv
Rows: 39177, Columns: 14
Columns: ['EA_CODE', 'PR_NAME', 'WardID', 'sal2023_est', 'centroid_lat', 'centroid_lng', 'euclidean_dist_m', 'euclidean_dist_km', 'walk_dist_m', 'walk_dist_km', 'drive_dist_m', 'drive_dist_km', 'walk_circuity', 'drive_circuity']


In [ ]:
NETWORK_DIR = "data/networks"
IMAGE_DIR   = "images/sal_pharmacy_distance"
os.makedirs(IMAGE_DIR, exist_ok=True)

network_files = {
    "gauteng_walk":        "network_gauteng_walk.graphml",
    "gauteng_drive":       "network_gauteng_drive.graphml",
    "kwazulu_natal_walk":  "network_kwazulu_natal_walk.graphml",
    "kwazulu_natal_drive": "network_kwazulu_natal_drive.graphml",
}

for key, filename in network_files.items():
    G = ox.load_graphml(os.path.join(NETWORK_DIR, filename))

    fig, ax = ox.plot_graph(
        G,
        figsize=(10, 10),
        node_size=0,
        edge_linewidth=0.3,
        edge_color="black",
        bgcolor=(0, 0, 0, 0),
        show=False,
        close=False,
    )

    ax.set_axis_off()
    fig.patch.set_alpha(0)

    save_path = os.path.join(IMAGE_DIR, f"network_{key}.png")
    fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent=True)

    fig.set_dpi(150)
    plt.show()
    plt.close(fig)
    print(f"Saved: {save_path}")

## NOTES

- SAL centroids are geometric (unweighted), and for irregularly shaped or large rural SALs, the centroid may fall in uninhabited areas, introducing measurement error. Population-weighted centroids using building footprints would reduce this bias.

- OSM road and pedestrian networks may have coverage gaps in rural and informal settlement areas, so missing roads mean the algorithm either routes through longer detours or marks centroids as unreachable. This is why thre is the "reachable" percentage in one of the outputs above for an indication of coverage quality.

- Drive networks are directed, meaning that one-way streets matter. The code converts to undirected for multi-source Dijkstra that greatly simplifies it. For more accurate drive distances it would be best to consider using directed shortest paths from each centroid to nearest pharmacy node, although this is computationally more expensive.

- The walk network includes sidewalks, footpaths, and pedestrian-accessible roads, but in practice pedestrians in informal settlements may use paths not mapped in OSM.

- The circuity is the ratio of network distance to Euclidean distance, which is a useful diagnostic. Ratios of 1.2-1.5 are typical for urban grids, and higher ratios suggest barriers (rivers, highways, rail lines) or sparse road networks. Extremely high ratios (>3) may indicate network data gaps.

- Both centroids and pharmacies are snapped to the nearest network node. For pharmacies near highways, this may snap to a highway node rather than the actual pedestrian-accessible entrance. The max snap distance is not capped in this implementation, however, adding a maximum snap tolerance (e.g. 500m) would flag pharmacies or centroids that are far from any mapped road.